# Red Team Evaluation - Static Prompt Set

Test a target with a pre-written set of attack prompts. Useful for regression testing and benchmarking against a fixed corpus instead of adaptive attacks.

Set `execution_strategy_type="static_prompt_set"` and provide a `prompt_set_id`. Built-in sets include:
- `do-anything-now-ape`
- `financial-assistant-basic`
- `financial-assistant-advanced`

**Prerequisites:**
- `pip install -r ../../requirements.txt`
- `HIDDENLAYER_CLIENT_ID` and `HIDDENLAYER_CLIENT_SECRET` in `../../.env`

**SDK Reference:** [hiddenlayer-sdk-python](https://github.com/hiddenlayerai/hiddenlayer-sdk-python)

## Setup

In [ ]:
import os

import httpx
from dotenv import load_dotenv
from hiddenlayer import AsyncHiddenLayer

load_dotenv("../../.env")

hl_client = AsyncHiddenLayer(
    client_id=os.getenv("HIDDENLAYER_CLIENT_ID"),
    client_secret=os.getenv("HIDDENLAYER_CLIENT_SECRET"),
)

## Configurables

1. Configure target
2. Configure prompt set
3. Configure evaluation

In [ ]:
# Target configuration
TARGET_URL = "https://api.example.com/v1/chat"
TARGET_MODEL_NAME = "custom-model"
TARGET_SYSTEM_PROMPT = "You are a helpful AI assistant."
CUSTOM_HEADERS = {
    # "Authorization": "Bearer your-api-key",
}

# Static prompt set configuration
PROMPT_SET_ID = "do-anything-now-ape"

# Evaluation configuration
EVAL_NAME = "Static Prompt Red Team Eval"  # Name of the evaluation
PARALLEL_TECHNIQUES = 10  # Options: 1-10
HIDDENLAYER_PROJECT_ID = None  # Options: None, <project_id>

## Verify payload adapters

- `to_target_request` needs to align with the target's request payload schema
- `from_target_response` needs to extract the reply text from the target's response payload schema

In [ ]:
def to_target_request(prompt, history, session_id):
    """Format the attack payload to match the target's request schema."""
    messages = [{"role": msg["role"], "content": msg["content"]} for msg in history]
    messages.append({"role": "user", "content": prompt})
    return {"model": TARGET_MODEL_NAME, "messages": messages}


def from_target_response(api_response):
    """Extract the response text from the target's response schema."""
    if "choices" in api_response:
        return api_response["choices"][0]["message"]["content"]
    return api_response.get("response", str(api_response))

## Create Handler

The handler acts as a proxy between the attacker and the target: it receives a static prompt from HiddenLayer, calls the target, and returns the target's reply.

In [ ]:
async def handler(prompt, history, session_id, target_system_prompt):
    """Handler acts as proxy between attacker and target."""
    payload = to_target_request(prompt, history, session_id)
    headers = {"Content-Type": "application/json", **CUSTOM_HEADERS}

    async with httpx.AsyncClient(timeout=None) as client:
        response = await client.post(TARGET_URL, json=payload, headers=headers)
        response.raise_for_status()
        return from_target_response(response.json())

## Run the Evaluation

Open a red team session that replays the static prompt set against the target.

In [ ]:
session = await hl_client.evaluation_sessions.red_team.start_session(
    name=EVAL_NAME,
    target_model=TARGET_MODEL_NAME,
    target_system_prompt=TARGET_SYSTEM_PROMPT,
    execution_strategy_type="static_prompt_set",
    prompt_set_id=PROMPT_SET_ID,
    max_parallel_techniques=PARALLEL_TECHNIQUES,
    hiddenlayer_project_id=HIDDENLAYER_PROJECT_ID,
)

print(f"Session started: {session.workflow_id}")
print(f"Testing prompts from: {PROMPT_SET_ID}")

await session.run_with_callback_parallel(handler=handler)

print("Evaluation complete. View transcripts at https://console.hiddenlayer.ai")

## Retrieve Results

Fetch and summarize the evaluation report for the completed session.

In [ ]:
results = await hl_client.evaluations.red_team.retrieve_evaluation_results(
    session.workflow_id
)

report = results.result.report
summary = report["summary"]

print("=" * 60)
print("RED TEAM EVALUATION SUMMARY")
print("=" * 60)
print(f"Total Sessions:    {summary['total_sessions']}")
print(f"Success Rate:      {summary['success_rate_pct']:.1f}%")
print(f"Successful:        {summary['success_total']} / {summary['attempts_total']}")
print(f"Errors:            {summary['error_total']}")